# Análisis de elasticidad-precio — Pizza Sales Dataset
**Curso:** Modelamiento Predictivo de Datos — Ingeniería Industrial
**Dataset:** [Pizza Sales Dataset (Kaggle)](https://www.kaggle.com/datasets/nextmillionaire/pizza-sales-dataset)

**Pregunta de negocio:** ¿cómo reacciona la cantidad demandada de cada variante de pizza ante
cambios en su precio, y cómo se puede usar esa relación para optimizar precios e ingresos?

**Etapa de la guía:** *Etapa 2 — reproducir el modelo en un notebook*.
Este notebook reproduce, en una versión reducida, el enfoque de estimación de
elasticidad-precio mediante regresión log-log, siguiendo la lógica metodológica de:

- Shuptar, S. (2022). *Prognostication model and pricing strategy optimization for
  restaurant businesses* [Bachelor's thesis, Ukrainian Catholic University].
- Petříček, M. et al. *Price Elasticity of Demand Measurement Using Log-Log Regression
  Analysis*. University College Prague.

> Antes de ejecutar: descarga `pizza_sales.csv` desde el enlace de Kaggle y colócalo en
> la carpeta `data/` del proyecto.

In [ ]:
import sys
sys.path.append("..")  # para poder importar utils_modelo.py desde /notebooks

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from utils_modelo import (
    cargar_datos, agregar_por_producto, estimar_elasticidad_global,
    estimar_elasticidad_por_categoria, interpretar_elasticidad,
    entrenar_modelo_predictivo, simular_cambio_precio,
)

sns.set_style("whitegrid")
RUTA_CSV = "../data/pizza_sales.csv"  # ajustar si el archivo tiene otro nombre

## 1. Carga y exploración inicial

In [ ]:
df = cargar_datos(RUTA_CSV)
print(f"Filas: {len(df):,} | Columnas: {df.shape[1]}")
df.head()

In [ ]:
df[["unit_price", "quantity", "total_price"]].describe()

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(12, 4))
sns.histplot(df["unit_price"], bins=20, ax=ax[0]).set_title("Distribución de precios unitarios")
sns.countplot(y=df["pizza_category"], order=df["pizza_category"].value_counts().index, ax=ax[1])
ax[1].set_title("Líneas de orden por categoría")
plt.tight_layout()
plt.show()

## 2. Agregación a nivel de producto

El precio de cada pizza es prácticamente constante a lo largo del año (no hay
experimentos de precio en el tiempo), por lo que la variación de precio que
podemos explotar es la que existe **entre variantes de pizza** (distintos
nombres y tamaños). Se agrega el dataset a nivel de `pizza_name_id` para
construir la curva de demanda cruzada.

In [ ]:
agg = agregar_por_producto(df)
print(f"Variantes de pizza analizadas: {len(agg)}")
agg.sort_values("cantidad_total", ascending=False).head(10)

In [ ]:
plt.figure(figsize=(6, 5))
sns.regplot(data=agg, x="ln_precio", y="ln_cantidad", ci=95,
            scatter_kws={"alpha": 0.5})
plt.title("Relación log-precio vs. log-cantidad (curva de demanda)")
plt.xlabel("ln(precio promedio)")
plt.ylabel("ln(cantidad total vendida)")
plt.show()

## 3. Estimación de la elasticidad-precio (regresión log-log)

Modelo: `ln(Q) = b0 + b1*ln(P) + categoría + tamaño`, donde **b1 es la elasticidad-precio
de la demanda** (interpretación directa en porcentaje: un aumento de 1% en el precio
se asocia a un cambio de b1% en la cantidad demandada).

In [ ]:
modelo_global, elasticidad_global = estimar_elasticidad_global(agg)
print(modelo_global.summary())
print()
print(f"Elasticidad-precio global estimada: {elasticidad_global:.3f}")
print(interpretar_elasticidad(elasticidad_global))

In [ ]:
tabla_categorias = estimar_elasticidad_por_categoria(agg)
tabla_categorias

In [ ]:
plt.figure(figsize=(6, 4))
sns.barplot(data=tabla_categorias, x="elasticidad", y="pizza_category")
plt.axvline(-1, color="red", linestyle="--", label="Umbral elástico/inelástico")
plt.legend()
plt.title("Elasticidad-precio estimada por categoría")
plt.show()

## 4. Modelo predictivo complementario (Random Forest)

Mientras el modelo log-log da un coeficiente de elasticidad interpretable, este
segundo modelo predice la cantidad esperada por línea de orden a partir del precio,
categoría, tamaño y estacionalidad (día de la semana, mes). Sirve como base para
comparar el efecto simulado del precio contra un modelo no lineal.

In [ ]:
modelo_rf, columnas_X, metricas = entrenar_modelo_predictivo(df)
print("Desempeño del modelo predictivo (conjunto de prueba):")
print(metricas)

## 5. Simulación de cambio de precio

Se usa la elasticidad estimada para simular el efecto de mover el precio de una
pizza en +/- X% sobre la cantidad demandada y el ingreso esperado. Esta lógica
es la que luego se traslada a la app de Streamlit (Etapa 4 de la guía).

In [ ]:
producto_ejemplo = agg.sort_values("cantidad_total", ascending=False).iloc[0]

resultado = simular_cambio_precio(
    precio_base=producto_ejemplo["precio_promedio"],
    cantidad_base=producto_ejemplo["cantidad_total"],
    elasticidad=elasticidad_global,
    variacion_pct=0.10,  # +10% de precio
)
resultado

## 6. Conclusiones y limitaciones

- La elasticidad estimada aquí es **de tipo cruzado/estructural** (entre variantes de
  pizza), no una elasticidad temporal pura, porque el dataset no registra cambios de
  precio en el tiempo para un mismo producto. Esta limitación coincide con la señalada
  por Shuptar (2022): al no existir experimentos reales de precio, se recurre a
  simulación a partir del coeficiente estimado.
- El modelo log-log asume una forma funcional log-lineal y controla solo por
  categoría y tamaño; podrían existir variables omitidas (ingredientes, estacionalidad,
  promociones) que afecten el resultado.
- El modelo Random Forest complementa el análisis pero no entrega un coeficiente de
  elasticidad directamente interpretable — se usa como referencia de desempeño predictivo.
- Próximo paso (Etapa 3 de la guía): convertir este notebook en un script `.py`
  modularizado (ya extraído en `utils_modelo.py`) y construir la interfaz en Streamlit
  (`app_streamlit.py`).